In [2]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
!nvidia-smi
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import  sklearn
from sklearn.model_selection import train_test_split
#import ray
import os
import math
import matplotlib.pyplot as plt
import tensorflow.keras.optimizers.schedules as schedules
import pickle 

Thu Nov 16 01:08:45 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 520.61.05    Driver Version: 520.61.05    CUDA Version: 11.8     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ...  On   | 00000000:05:00.0 Off |                  N/A |
|  0%   30C    P5    10W / 198W |      1MiB /  8192MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA GeForce ...  On   | 00000000:06:00.0 Off |                  N/A |
| 30%   

In [3]:
os.cpu_count()

12

In [4]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)


In [5]:
#ray.shutdown()
#ray.init(num_cpus=2, num_gpus=0)
#ray.shutdown()

# Data processing

For the homework, you should download the dataset with the provided link. The provided dataset has already partitioned the data by clients.

In [6]:
NUM_CLIENT = 100
train_data = 'train_data.npy'
train_data_arr = np.load(train_data, allow_pickle=True)
test_data = 'test_data.npy'
test_data_arr = np.load(test_data, allow_pickle=True)

# Define model

In [7]:
## regular model 
'''
def get_model():
    model = tf.keras.models.Sequential(
        [ tf.keras.layers.Flatten(input_shape=(28, 28)),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(62, activation="softmax")])
    model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    return model
'''
## adding leanring rate.
def get_model(lr):
    lr_schedule = schedules.ExponentialDecay(
        initial_learning_rate=lr,
        decay_steps=10000,
        decay_rate=0.9,
        staircase=True
    )

    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(input_shape=(28, 28)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(62, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


'''
## normalized data 
def get_model():
    model = tf.keras.models.Sequential(
        [ tf.keras.layers.Lambda(lambda x: x / 255.0, input_shape=(28, 28)),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(62, activation="softmax")])
    model.compile("adam", "sparse_categorical_crossentropy", metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])
    return model
'''

'\n## normalized data \ndef get_model():\n    model = tf.keras.models.Sequential(\n        [ tf.keras.layers.Lambda(lambda x: x / 255.0, input_shape=(28, 28)),\n            tf.keras.layers.Flatten(),\n            tf.keras.layers.Dense(128, activation="relu"),\n            tf.keras.layers.Dense(128, activation="relu"),\n            tf.keras.layers.Dense(62, activation="softmax")])\n    model.compile("adam", "sparse_categorical_crossentropy", metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])\n    return model\n'

## Splitting Data

In [8]:
all_train = []
all_val = []
x_train_all = []
y_train_all = []
x_val_all =[]
y_val_all =[]

for i in range(100):
    
    data = train_data_arr[i]
    # Split the data into training and validation sets
    images_train, images_val, labels_train, labels_val = train_test_split(
        data['images'],
        data['labels'],
        test_size=0.2,
        random_state=42)
    # Normalize the input data
    #images_train = np.array(images_train)
    #images_val = np.array(images_val)
    #images_train, images_val = images_train / 255.0, images_val / 255.0
    
    
    # Create TensorFlow datasets
    #expand dimensions of input to (1, 28, 28):
    #print("images before expansion: ", images_train.shape)
    #images_train = np.array([np.expand_dims(image, axis=0) for image in images_train])
    #images_val = [np.expand_dims(image, axis=0) for image in images_val]
    #images_train = np.expand_dims(images_train, axis=1)  # Add batch dimension
    #images_val = np.expand_dims(images_val, axis=1)
    #print("images expanded: ", images_train[0].shape)
    train_dataset = tf.data.Dataset.from_tensor_slices((images_train, labels_train))
    val_dataset = tf.data.Dataset.from_tensor_slices((images_val, labels_val))
    all_train.append(train_dataset)
    all_val.append(val_dataset)


2023-11-16 01:08:46.809085: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-11-16 01:08:48.868395: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7377 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1080, pci bus id: 0000:05:00.0, compute capability: 6.1
2023-11-16 01:08:48.868936: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 7377 MB memory:  -> device: 1, name: NVIDIA GeForce GTX 1080, pci bus id: 0000:06:00.0, compute capability: 6.1
2023-11-16 01:08:48.869408: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/de

# Client class

In [9]:
#@ray.remote(num_cpus=1, num_gpus=0)
class Client:
    def __init__(self, i, batch_size, initial_learning_rate, E):
        # Create model
        self.model = get_model(initial_learning_rate)
        self.training_data = all_train[i]
        self.validation_data = all_val[i]
        self.batch_size = batch_size
        self.E = E

    def train(self, global_weights):
        self.model.set_weights(global_weights)
        
        
        # Define a custom callback for printing validation metrics every 10 epochs
        '''
        class PrintValidationMetrics(tf.keras.callbacks.Callback):
            def on_epoch_end(self, epoch, logs=None):
                if epoch % 10 == 0:
                    val_loss, val_accuracy = self.model.evaluate(
                        self.validation_data.batch(self.batch_size), verbose=0)
                    print(f'Epoch {epoch}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')

        callback = PrintValidationMetrics()
        
        '''
        
        self.model.fit(self.training_data.batch(self.batch_size), epochs=self.E, validation_data=self.validation_data.batch(self.batch_size), verbose=0)
        ### to get accuracy and loss as arrays for all epochs: 
        '''
         history = self.model.fit(self.training_data.batch(self.batch_size), epochs=E, validation_data=self.validation_data.batch(self.batch_size), verbose=0)
        
        # Access the training history
        ## define objects of these arrays in constructor
        train_loss = history.history['loss']
        train_accuracy = history.history['accuracy']
        val_loss = history.history['val_loss']
        val_accuracy = history.history['val_accuracy']

        return train_loss, train_accuracy, val_loss, val_accuracy

        '''
        return self.model.get_weights()

    def evaluate(self):
        #gives scalars of loss and accuracy at the end of training 
        train_loss, train_accuracy = self.model.evaluate(self.training_data.batch(self.batch_size), verbose=0)
        val_loss, val_accuracy = self.model.evaluate(self.validation_data.batch(self.batch_size))
        return train_loss, train_accuracy, val_loss, val_accuracy


# Aggregated metrics and losses
Aggregate client metrics and losses for (federated) evaluation metrics.

In [10]:
sample_nbs = []
for i in range(100):
    #print(i)
    #print(len(train_data_arr[i]['labels']))
    #print("train data: ", len( all_train[i]))
    #print("80% of local data: ", len(train_data_arr[i]['labels'])*0.8)
    sample_nbs.append(len(train_data_arr[i]['labels']))
#sample_nbs

In [11]:
def weighted_average(sampled_clients, metric):
    # TODO: Aggregate client losses or accuracies by taking a weighted average.
    #       Weights are proportional to the number of samples at each client
    sample_size_selected = [sample_nbs[i] for i in sampled_clients]
    #sorted list so order should match
    agg_train_acc = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['train_accuracy'])) / sum(sample_size_selected)
    agg_train_loss = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['train_loss'])) / sum(sample_size_selected)
    agg_val_acc = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['val_accuracy'])) / sum(sample_size_selected)
    agg_val_loss = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['val_loss'])) / sum(sample_size_selected)


    return agg_train_acc, agg_train_loss, agg_val_acc, agg_val_loss

# Aggregation of client updates

In [12]:
def fedAvg(sampled_clients, client_weights):
    # Get the size of samples for each client
    sample_size_selected = [sample_nbs[c] for c in sampled_clients]
    #print("shape of local weights: ", len(client_weights[0]), client_weights[0][0].shape)
    list_dim = len(client_weights[0])
    np_dim = client_weights[0][0].shape
    total = sum(sample_size_selected)
    
    avg_weights = []
    for k in range(list_dim):
        avg_weights.append(np.zeros(client_weights[0][k].shape))

    for i in range (len(sampled_clients)):
        scalar = sample_size_selected[i]/total
        for j in range(0, list_dim):
            this_numpy = np.multiply(client_weights[i][j], scalar)
            avg_weights[j] = np.add(avg_weights[j],this_numpy)  
    #print("shape of avg weights: ", len(avg_weights), avg_weights[0].shape)
    return avg_weights


In [13]:
def plots_train(normalized, t, client, E, batch_size,  train_loss, train_accuracy, val_loss, val_accuracy):
    plt.figure(figsize=(12, 4))
    #plt.subplot(1, 2, 1)
    plt.plot(train_loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Rounds')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(normalized + '_train_loss_'+str(t)+'_'+str(client)+'_'+str(E)+'_'+str(batch_size)+'_.png')
    plt.close()
    # Plot training accuracy and validation accuracy
    plt.figure(figsize=(12, 4))
    plt.plot(train_accuracy, label='Training Accuracy')
    plt.plot(val_accuracy, label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Rounds')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig(normalized + '_train_accuracy_'+str(t)+'_'+str(client)+'_'+str(E)+'_'+str(batch_size)+'_.png')
    plt.close()

# Launch the learning

In [14]:
run = "p1-new_run2"
Agg_metric = {}
Agg_metric['train_loss'] = []
Agg_metric['train_accuracy'] =[]
Agg_metric['val_loss'] =[]
Agg_metric['val_accuracy'] =[]
    
# Initialize the global model weights
batch_size = 32
# Initialize the global model weights with an initial learning rate
initial_learning_rate = 0.001
global_model = get_model(initial_learning_rate)

#global_model = get_model()
users = np.arange(100)
# Get data
#data_by_clients, test_data = partition_data()

total_rounds = 500

C = 10   # Only a fraction of C of clients participate in the local training per round
E = 20     # Number of epochs for each local training

for round in range(total_rounds):
    # Step 1
    print("communication round: ", round)
    #initiate empty list of model weights 
    client_weights =[]
    metric = {}
    metric['train_loss'] = []
    metric['train_accuracy'] =[]
    metric['val_loss'] =[]
    metric['val_accuracy'] =[] 
    # TODO: Sample C fraction of clients
    sampled_clients = np.random.choice(users, C, replace=False)
    sampled_clients = sorted(sampled_clients)
    # TODO: Get the local data for the selected clients
    for i in sampled_clients:
        print("client: ", i)
        #perform local training
        #define instance of client:
        current_client = Client(i, batch_size, initial_learning_rate, E) 
        
        #save local weights
        
        local_weights = current_client.train(global_model.get_weights())
        #print(local_weights[0].shape)
        client_weights.append(local_weights)
        #save local results
        
        train_loss, train_accuracy, val_loss, val_accuracy = current_client.evaluate()
        metric['train_loss'].append(train_loss)
        metric['train_accuracy'].append(train_accuracy)
        metric['val_loss'].append(val_loss)
        metric['val_accuracy'].append(val_accuracy)
        #print("train_loss : ", train_loss)
        #print("train_accuracy: ", train_accuracy)
        

    # Step 3
    # TODO: Aggregate the losses and metrics, keep track of the metrics
    agg_train_acc, agg_train_loss, agg_val_acc, agg_val_loss = weighted_average(sampled_clients, metric)
    Agg_metric['train_loss'].append(agg_train_loss)
    Agg_metric['train_accuracy'].append(agg_train_acc)
    Agg_metric['val_loss'].append(agg_val_loss)
    Agg_metric['val_accuracy'].append(agg_val_acc)
    print("aggregated validation accuracy: ",Agg_metric['val_accuracy'][round]) 
    # Step 4
    # TODO: Aggregate the client updates
    weighted_avg_weights = fedAvg(sampled_clients, client_weights)
    
    # Step 5
    # Update the global model
    global_model.set_weights(weighted_avg_weights)
    
    if round%50==0 and round!=0:


    # Specify the file path
        file_path = 'Agg_metric_run3_'+str(round)+'_'+str(E)+'_'+str(C)+'_'+str(batch_size)+'_.pickle'

        # Save the dictionary to a pickle file
        with open(file_path, 'wb') as file:
            pickle.dump(Agg_metric, file)

        print(f'Dictionary saved to {file_path}')

        global_model.save('global_model_'+str(round)+'_'+str(E)+'_'+str(C)+'_'+str(batch_size)+'_.h5')
        plots_train(run, round, C, E, batch_size,  Agg_metric['train_loss'] , Agg_metric['train_accuracy'], Agg_metric['val_loss'], Agg_metric['val_accuracy'])
        

communication round:  0
client:  28
3/3 [==============================] - 0s 2ms/step - loss: 3.4934 - accuracy: 0.3896
client:  31
3/3 [==============================] - 0s 2ms/step - loss: 3.0791 - accuracy: 0.4308
client:  34
3/3 [==============================] - 0s 1ms/step - loss: 3.4670 - accuracy: 0.2174
client:  39
2/2 [==============================] - 0s 2ms/step - loss: 3.5122 - accuracy: 0.2963
client:  51
3/3 [==============================] - 0s 2ms/step - loss: 3.3690 - accuracy: 0.1233
client:  74
2/2 [==============================] - 0s 2ms/step - loss: 3.2123 - accuracy: 0.3667
client:  78
3/3 [==============================] - 0s 2ms/step - loss: 3.6735 - accuracy: 0.1493
client:  93
2/2 [==============================] - 0s 2ms/step - loss: 4.3899 - accuracy: 0.0877
client:  94
3/3 [==============================] - 0s 2ms/step - loss: 3.5220 - accuracy: 0.3077
client:  99
3/3 [==============================] - 0s 2ms/step - loss: 3.2837 - accuracy: 0.2571
aggreg

2023-11-16 01:23:40.736998: W tensorflow/core/data/root_dataset.cc:266] Optimization loop failed: CANCELLED: Operation was cancelled


3/3 [==============================] - 0s 2ms/step - loss: 1.2086 - accuracy: 0.7286
aggregated validation accuracy:  0.6772802133331339
communication round:  116
client:  10
3/3 [==============================] - 0s 2ms/step - loss: 1.2311 - accuracy: 0.8026
client:  13
3/3 [==============================] - 0s 2ms/step - loss: 1.1196 - accuracy: 0.7027
client:  23
2/2 [==============================] - 0s 2ms/step - loss: 2.0467 - accuracy: 0.6333
client:  24
2/2 [==============================] - 0s 2ms/step - loss: 1.5127 - accuracy: 0.5833
client:  26
3/3 [==============================] - 0s 2ms/step - loss: 2.0602 - accuracy: 0.6761
client:  49
2/2 [==============================] - 0s 2ms/step - loss: 2.0124 - accuracy: 0.6833
client:  54
3/3 [==============================] - 0s 2ms/step - loss: 1.5500 - accuracy: 0.7083
client:  66
2/2 [==============================] - 0s 2ms/step - loss: 2.2820 - accuracy: 0.6250
client:  72
2/2 [==============================] - 0s 2ms/ste

2023-11-16 02:08:10.620751: W tensorflow/core/data/root_dataset.cc:266] Optimization loop failed: CANCELLED: Operation was cancelled


2/2 [==============================] - 0s 2ms/step - loss: 2.5387 - accuracy: 0.6098
aggregated validation accuracy:  0.6460280522649359
communication round:  442
client:  3
3/3 [==============================] - 0s 2ms/step - loss: 1.4861 - accuracy: 0.7101
client:  15
3/3 [==============================] - 0s 2ms/step - loss: 2.0245 - accuracy: 0.6029
client:  16
3/3 [==============================] - 0s 2ms/step - loss: 1.2922 - accuracy: 0.6667
client:  19
2/2 [==============================] - 0s 2ms/step - loss: 1.2366 - accuracy: 0.6508
client:  21
3/3 [==============================] - 0s 2ms/step - loss: 1.3315 - accuracy: 0.6471
client:  25
1/1 [==============================] - 0s 7ms/step - loss: 5.7539 - accuracy: 0.5833
client:  38
3/3 [==============================] - 0s 2ms/step - loss: 2.4361 - accuracy: 0.5942
client:  44
2/2 [==============================] - 0s 3ms/step - loss: 1.7057 - accuracy: 0.6786
client:  51
3/3 [==============================] - 0s 2ms/step

In [15]:
##loading saved model
'''
# Load the model from the saved file
loaded_model = tf.keras.models.load_model('my_model.h5')

# Now you can use loaded_model for predictions or further training
import tensorflow as tf

# Load the saved model
loaded_model = tf.keras.models.load_model('my_model.h5')

# Get additional training data
additional_data = ...

# Compile the model if needed (only if you did not save the optimizer state)
loaded_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Continue training with additional data
loaded_model.fit(additional_data, epochs=additional_epochs)

# Save the model with updated weights
loaded_model.save('my_model_updated.h5')

print('Updated model saved to my_model_updated.h5')


'''

"\n# Load the model from the saved file\nloaded_model = tf.keras.models.load_model('my_model.h5')\n\n# Now you can use loaded_model for predictions or further training\nimport tensorflow as tf\n\n# Load the saved model\nloaded_model = tf.keras.models.load_model('my_model.h5')\n\n# Get additional training data\nadditional_data = ...\n\n# Compile the model if needed (only if you did not save the optimizer state)\nloaded_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])\n\n# Continue training with additional data\nloaded_model.fit(additional_data, epochs=additional_epochs)\n\n# Save the model with updated weights\nloaded_model.save('my_model_updated.h5')\n\nprint('Updated model saved to my_model_updated.h5')\n\n\n"

# Create plots

In [16]:
def test(test_data_arr, weights, batch_size, lr):
    model = get_model(lr)  # Construct the model
    model.set_weights(weights)  # Set model weights with the latest parameters

    # Assuming test_data is a tuple of test_images and test_labels
    test_images = np.array(test_data_arr[0]['images'])
    test_labels = np.array(test_data_arr[0]['labels'])
    
 

    # Normalize the input data
    #test_images = np.array(test_images)
    #test_images= test_images / 255.0
    
    test_images_flat = test_images.reshape(test_images.shape[0], -1)
    
    test_dataset = tf.data.Dataset.from_tensor_slices((test_images, test_labels))
    batch_size = batch_size  # Set your desired batch size
    # Evaluate the model using the dataset
    batched_test_dataset = test_dataset.batch(batch_size)

    test_loss, test_accuracy = model.evaluate(batched_test_dataset)


    print(f"Test Accuracy: {test_accuracy:.4f}")

    return test_loss, test_accuracy

In [17]:
plots_train(run, total_rounds, C, E, batch_size, Agg_metric['train_loss'] , Agg_metric['train_accuracy'], Agg_metric['val_loss'], Agg_metric['val_accuracy'])
# Evaluate on test data after training
test_loss, test_accuracy = test(test_data_arr, global_model.get_weights(), batch_size, initial_learning_rate)

114/114 [==============================] - 0s 1ms/step - loss: 2.1894 - accuracy: 0.5206
Test Accuracy: 0.5206


In [18]:
print("test loss: ", test_loss)
print("test accuracy: ", test_accuracy)

test loss:  2.18941330909729
test accuracy:  0.5205744504928589


In [19]:
test_results = np.array ([test_loss, test_accuracy])
np.save('test_results_'+run+'_'+str(total_rounds)+'_'+str(C)+'_'+str(E)+'_'+str(batch_size)+'_.npy', test_results)

In [20]:
##load pickle file 
'''
# Load the dictionary from the pickle file
with open(file_path, 'rb') as file:
    loaded_dict = pickle.load(file)

print('Loaded dictionary:', loaded_dict)
'''

"\n# Load the dictionary from the pickle file\nwith open(file_path, 'rb') as file:\n    loaded_dict = pickle.load(file)\n\nprint('Loaded dictionary:', loaded_dict)\n"